# 🌲 3-Hour Grand Meditative Nature Film (Colab GPU Renderer)

This notebook renders the **3-Hour Ultra-HD 1080p Master Film** combining:
- **Music**: All 28 original compositions slowed by 10% (`atempo=0.90`) for deep mindfulness
- **Visuals**: **ALL 42 background nature videos** from your `Instrumentals/` folder at **50% slow-motion** (`setpts=2.0*PTS`)
- **Sound Bed**: Real ambient nature soundscape (rivers, rain, ocean waves, bamboo breeze) mixed softly at 25% volume
- **Hardware Acceleration**: High-speed NVIDIA GPU `h264_nvenc` encoding

In [ ]:
# Step 1: Verify NVIDIA GPU
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Extract Assets & Prepare Environment
import os, glob, subprocess, math, time

DRIVE_DIR = "/content/drive/MyDrive/Meditative_3Hour_Production"
LOCAL_WORK = "/content/render_work"
os.makedirs(LOCAL_WORK, exist_ok=True)

# Untar the 42 nature videos archive
tar_path = os.path.join(DRIVE_DIR, "nature_videos.tar")
videos_dir = os.path.join(LOCAL_WORK, "videos")
os.makedirs(videos_dir, exist_ok=True)

print("📦 Extracting all 42 nature video clips...")
!tar -xf "$tar_path" -C "$videos_dir"

clips = sorted(glob.glob(os.path.join(videos_dir, "*.mp4")))
print(f"✅ Extracted {len(clips)} nature video clips!")
for c in clips[:5]:
    print("  •", os.path.basename(c))
print(f"  ... and {len(clips)-5} more")

audio_path = os.path.join(DRIVE_DIR, "Master_3Hour_Meditative_Music.wav")
print("🎵 3-Hour Master Audio located:", audio_path, f"({os.path.getsize(audio_path)/1024/1024:.1f} MB)")

In [ ]:
# Step 4: Run High-Speed GPU Master Film Assembly
import subprocess, math, time

output_mp4 = os.path.join(DRIVE_DIR, "Master_3Hour_Meditative_Film_1080p.mp4")
total_dur = 10800.0  # Exactly 3 Hours (3 * 3600)

# 42 clips * 20s (50% slow-mo) = 840s per full cycle
# Loop the 42 clips sequentially to cover 3 hours seamlessly
loops = math.ceil((total_dur + 60) / (len(clips) * 20.0))
concat_list = os.path.join(LOCAL_WORK, "concat_3h.txt")
with open(concat_list, "w") as f:
    for _ in range(loops):
        for cp in clips:
            f.write(f"file '{cp}'\n")

print(f"🚀 Starting NVIDIA NVENC 1080p GPU Render for 3-Hour Master Film...")
render_cmd = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_list,
    "-i", audio_path,
    "-filter_complex",
    "[0:v]setpts=2.0*PTS,scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,fps=30[vout];"
    "[0:a]atempo=0.5,volume=0.25[a_amb];"
    "[1:a][a_amb]amix=inputs=2:duration=first:dropout_transition=3[aout]",
    "-map", "[vout]", "-map", "[aout]",
    "-c:v", "h264_nvenc", "-preset", "p4", "-rc", "vbr", "-cq", "21", "-b:v", "4500k",
    "-c:a", "aac", "-b:a", "320k", "-ar", "48000",
    "-movflags", "+faststart",
    "-t", str(total_dur),
    output_mp4
]

t0 = time.time()
subprocess.run(render_cmd, check=True)
print(f"🎉 SUCCESS! 3-Hour Meditative Film completed in {(time.time()-t0)/60:.1f} minutes!")
print(f"Saved to: {output_mp4}")